# Logit Lens Tutorial

This tutorial shows how to visualize the **logit lens** of transformer language models using NDIF for remote execution.

## What is the Logit Lens?

The logit lens is an interpretability technique that decodes hidden states at each layer into vocabulary probabilities. By applying the model's output projection (`norm` → `lm_head`) to intermediate layers, we can see how the model's predictions evolve through its computation.

## Setup

First, install the required packages:

In [ ]:
# Install from kitwidget branch with nnsight batch dimension fix
!pip install -q nnsight "interp-workbench @ git+https://github.com/davidbau/workbench.git@kitwidget"

## Configure NDIF

To run large models like Llama-8B remotely on NDIF, you need an API key.

1. Get a free API key at [nnsight.net](https://nnsight.net)
2. In Colab: Click the key icon in the left sidebar, add a secret named `NDIF_API`
3. Or set it as an environment variable

In [ ]:
import os
from nnsight import LanguageModel, CONFIG

# Try to get NDIF key from Colab secrets, then environment
try:
    from google.colab import userdata
    NDIF_API = userdata.get('NDIF_API')
except:
    NDIF_API = os.environ.get('NDIF_API')

if NDIF_API:
    CONFIG.set_default_api_key(NDIF_API)
    print("NDIF API key configured!")
else:
    print("Warning: No NDIF_API found. Remote execution will not work.")
    print("Add NDIF_API to Colab secrets or set NDIF_API environment variable.")

---

# Part 1: Using the LogitLens API

The simplest way to collect and visualize logit lens data:

In [ ]:
from nnsight import LanguageModel
from workbench.logitlens.collect import collect_logit_lens
from workbench.logitlens.display import show_logit_lens

# Load Llama-3.1-8B (runs remotely on NDIF)
model = LanguageModel("meta-llama/Llama-3.1-8B", device_map="auto")

# Collect logit lens data
data = collect_logit_lens(
    "The capital of France is",
    model,
    k=5,           # Track top 5 predictions per layer
    remote=True,   # Run on NDIF server
)

# Display interactive visualization
show_logit_lens(data, title="Llama-3.1-8B: The capital of France is")

### Interacting with the Widget

- **Click cells** to see top-k predictions at that layer/position
- **Click tokens** in the popup to pin their probability trajectories
- **Click input tokens** (left column) to compare multiple positions
- **Drag edges** to resize columns and chart

### Understanding the Data

Let's examine what `collect_logit_lens` returns:

In [ ]:
print("Keys:", list(data.keys()))
print()
print("model:", data["model"])
print("input:", data["input"])
print("layers:", data["layers"][:5], "...", f"({len(data['layers'])} total)")
print()
print("topk shape:", data["topk"].shape, "- [n_layers, n_positions, k]")
print("tracked[0] shape:", data["tracked"][0].shape, "- unique tokens at position 0")
print("probs[0] shape:", data["probs"][0].shape, "- [n_layers, n_tracked] trajectories")
print()
print("vocab (sample):", dict(list(data["vocab"].items())[:5]))

---

# Part 2: Collecting Logit Lens Data "By Hand"

To understand what's happening under the hood, let's implement logit lens collection manually using nnsight directly. This shows exactly how to access model internals.

## Step 1: Understand the Model Structure

Different models have different internal structures. For Llama models:

In [ ]:
# Llama model structure:
# - model.model.layers[i]  -> transformer layers
# - model.model.norm       -> final RMSNorm
# - model.lm_head          -> output projection to vocabulary

print(f"Model type: {model.config.model_type}")
print(f"Number of layers: {model.config.num_hidden_layers}")
print(f"Vocabulary size: {model.config.vocab_size}")

## Step 2: Manual Logit Lens Collection

Here's the complete implementation showing exactly what happens:

In [ ]:
import torch

def collect_logit_lens_by_hand(prompt, model, k=5, remote=True):
    """
    Collect logit lens data manually using nnsight.
    
    This shows exactly how to:
    1. Access hidden states at each layer
    2. Apply the final norm and lm_head
    3. Extract top-k predictions and trajectories
    """
    # Tokenize the prompt
    token_ids = model.tokenizer.encode(prompt)
    n_pos = len(token_ids)
    n_layers = model.config.num_hidden_layers
    
    # Access model components BEFORE trace context (Llama-specific paths)
    # This avoids serialization issues with remote execution
    layers = model.model.layers      # The transformer layers
    norm = model.model.norm          # Final RMSNorm  
    lm_head = model.lm_head          # Output projection
    
    # Run the model with tracing
    # When remote=True, computation happens on NDIF server
    with model.trace(token_ids, remote=remote):
        all_probs = []
        all_topk = []
        
        for layer_idx in range(n_layers):
            # Get the hidden state output from this layer
            # layers[i].output is a tuple; [0] is the hidden state
            hidden = layers[layer_idx].output[0]
            
            # Apply logit lens: norm -> lm_head -> softmax
            normed = norm(hidden)
            logits = lm_head(normed)
            
            # Handle batch dimension: local has [batch, pos, vocab],
            # remote has [pos, vocab]. Squeeze if needed.
            if logits.dim() == 3:
                logits = logits.squeeze(0)
            probs = torch.softmax(logits, dim=-1)
            
            # Store probabilities and top-k indices for this layer
            all_probs.append(probs)
            all_topk.append(probs.topk(k, dim=-1).indices.to(torch.int32))
        
        # Save individual layer results - stacking happens client-side
        result = {"all_topk": all_topk, "all_probs": all_probs}.save()
    
    # Client-side: stack results and compute tracked tokens
    topk = torch.stack(result["all_topk"], dim=0)  # [n_layers, n_pos, k]
    all_probs = result["all_probs"]  # List of [n_pos, vocab_size]
    
    # For each position: find unique tokens across all layers
    tracked = []
    probs_out = []
    for pos in range(n_pos):
        # Union of all tokens appearing in top-k at any layer
        unique = torch.unique(topk[:, pos, :].flatten()).to(torch.int32)
        # Extract probability trajectory for each unique token
        traj = torch.stack([all_probs[li][pos, unique] for li in range(n_layers)])
        tracked.append(unique)
        probs_out.append(traj)
    
    # Build vocabulary map (runs locally after server computation)
    all_ids = set(topk.flatten().tolist())
    for t in tracked:
        all_ids.update(t.tolist())
    vocab = {i: model.tokenizer.decode([i]) for i in all_ids}
    
    return {
        "model": model.config._name_or_path,
        "input": [model.tokenizer.decode([t]) for t in token_ids],
        "layers": list(range(n_layers)),
        "topk": topk,
        "tracked": tracked,
        "probs": probs_out,
        "vocab": vocab,
    }

In [ ]:
# Test our manual implementation
data_manual = collect_logit_lens_by_hand(
    "The Eiffel Tower is located in",
    model,
    k=5,
    remote=True,
)

# Visualize it
show_logit_lens(data_manual, title="Manual collection: The Eiffel Tower is located in")

## Key Concepts

### 1. Model Component Access
```python
# For Llama/Mistral/Qwen:
hidden = model.model.layers[i].output[0]  # Layer i output
normed = model.model.norm(hidden)          # Final norm
logits = model.lm_head(normed)             # Project to vocab

# For GPT-2/GPT-J:
hidden = model.transformer.h[i].output[0]
normed = model.transformer.ln_f(hidden)
logits = model.lm_head(normed)
```

### 2. Remote Execution
When `remote=True`, all computation inside `model.trace()` runs on NDIF servers. Only the `.save()`d tensors are sent back.

### 3. Bandwidth Optimization
Instead of sending full logits (~500MB for 70B model), we send only:
- Top-k indices: ~40KB
- Tracked trajectories: ~100KB

**1000x reduction!**

---

# Part 3: Analyzing Specific Layers

You can analyze a subset of layers for faster exploration:

In [ ]:
# Analyze every 4th layer (faster, still informative)
n_layers = model.config.num_hidden_layers
layer_subset = list(range(0, n_layers, 4))  # [0, 4, 8, 12, ...]

data_subset = collect_logit_lens(
    "1 + 1 =",
    model,
    k=5,
    layers=layer_subset,
    remote=True,
)

print(f"Analyzed layers: {data_subset['layers']}")
show_logit_lens(data_subset, title="Layer subset: 1 + 1 =")

---

# Part 4: Try Different Prompts

Explore how different prompts reveal different model behaviors:

In [ ]:
prompts = [
    "The quick brown fox jumps over the",
    "To be or not to be, that is the",
    "def fibonacci(n):\n    if n <=",
]

for prompt in prompts:
    data = collect_logit_lens(prompt, model, k=5, remote=True)
    display(show_logit_lens(data, title=prompt))

---

# Summary

You've learned:

1. **What the logit lens is**: Projecting intermediate hidden states to vocabulary probabilities

2. **How to use the library**: `collect_logit_lens()` + `show_logit_lens()`

3. **How it works internally**: Manual implementation showing nnsight layer access

4. **Why it's efficient**: Server-side top-k extraction reduces bandwidth by 1000x

## Next Steps

- Try larger models like `meta-llama/Llama-3.1-70B`
- Explore rank trajectories with `include_rank=True`
- Check the [LogitLens module docs](https://github.com/davidbau/workbench/blob/main/workbench/logitlens/README.md)